# Deploy a private Gemma 4 endpoint for `ask`

Deploy `google/gemma-4-E4B-it` as a private Hugging Face Inference Endpoint running vLLM. The notebook verifies its OpenAI-compatible Responses API, then prints the non-secret settings needed by `ask`.

Use it for short sessions and pause the endpoint when finished.

## Authenticate

Create a Hugging Face token that can manage Inference Endpoints and read the Gemma repository.

In [ ]:
import os

from huggingface_hub import HfApi

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("Set HF_TOKEN before starting Jupyter, then restart the kernel.")

api = HfApi(token=HF_TOKEN)
account = api.whoami()
print(f"Authenticated as {account['name']}")

## Configure the endpoint

This example deploys the upstream Gemma repository.

vLLM exposes `/v1/responses`. `MODEL_NAME` is the stable name sent by `ask` as `ASK_MODEL`.

In [ ]:
MODEL_REPOSITORY = f"{account['name']}/gemma-4-e4b-it-ask"
MODEL_REVISION = api.model_info(MODEL_REPOSITORY).sha
print(f"Deploying pinned revision: {MODEL_REVISION}")
ENDPOINT_NAME = "gemma-4-e4b-it-ask"
MODEL_NAME = "gemma-4-e4b-it-ask"

VENDOR = "aws"
REGION = "us-east-1"
INSTANCE_TYPE = "nvidia-l4"
INSTANCE_SIZE = "x1"
SCALE_TO_ZERO_TIMEOUT_MINUTES = 15

## Start or resume the endpoint

Run this cell to start a test session. It creates the endpoint if needed, resumes it if paused, and otherwise reuses it. The selected Hub model is mounted into the vLLM container.

In [ ]:
endpoints = {endpoint.name: endpoint for endpoint in api.list_inference_endpoints()}
endpoint = endpoints.get(ENDPOINT_NAME)

# Work around https://github.com/vllm-project/vllm/issues/44788.
# Removes unused normalization modules from KV-shared layers before starting the native Gemma 4 implementation.
VLLM_STARTUP_SCRIPT = f"""set -euo pipefail
python3 -m pip install --no-deps --disable-pip-version-check transformers==5.14.1
python3 - <<'PY'
from pathlib import Path

path = Path('/usr/local/lib/python3.12/dist-packages/vllm/model_executor/models/gemma4.py')
source = path.read_text()
needle = '''        self.rotary_emb = get_rope(
'''
replacement = '''
        if self.is_kv_shared_layer:
            del self.k_norm
            del self.v_norm

        self.rotary_emb = get_rope(
'''

if replacement not in source:
    if source.count(needle) != 1:
        raise RuntimeError('Pinned vLLM Gemma 4 source no longer matches the deployment patch.')
    path.write_text(source.replace(needle, replacement, 1))
PY
exec vllm serve /repository \
  --served-model-name {MODEL_NAME} \
  --host 0.0.0.0 \
  --port 8000 \
  --max-model-len 8192 \
  --gpu-memory-utilization 0.9 \
  --max-num-seqs 4 \
  --enforce-eager \
  --enable-auto-tool-choice \
  --reasoning-parser gemma4 \
  --tool-call-parser gemma4 \
  --chat-template /repository/chat_template.jinja
"""

endpoint_settings = dict(
        repository=MODEL_REPOSITORY,
        revision=MODEL_REVISION,
        framework="custom",
        task="text-generation",
        accelerator="gpu",
        instance_type=INSTANCE_TYPE,
        instance_size=INSTANCE_SIZE,
        min_replica=0,
        max_replica=1,
        scale_to_zero_timeout=SCALE_TO_ZERO_TIMEOUT_MINUTES,
        custom_image={"url": "vllm/vllm-openai:v0.27.0", "healthRoute": "/health", "port": 8000},
        container_command=["/bin/bash", "-lc"],
        container_args=[VLLM_STARTUP_SCRIPT],
        token=HF_TOKEN,
    )

if endpoint is None:
    endpoint = api.create_inference_endpoint(
        name=ENDPOINT_NAME, vendor=VENDOR, region=REGION, type="private", **endpoint_settings
    )
    print(f"Created {endpoint.name} ({endpoint.status})")
else:
    endpoint = api.update_inference_endpoint(name=ENDPOINT_NAME, **endpoint_settings)
    print(f"Updated {endpoint.name} ({endpoint.status})")

if endpoint.status == "paused":
    endpoint.resume()
    print(f"Resuming {endpoint.name}")

In [ ]:
print("Waiting up to 10 minutes for the endpoint to become healthy...")
endpoint.wait(timeout=10*60, refresh_every=15)

BASE_URL = endpoint.url.rstrip("/") + "/v1"
print(f"Endpoint ready: {BASE_URL}")

## Verify the Responses API

In [ ]:
import json

from openai import OpenAI

client = OpenAI(api_key=HF_TOKEN, base_url=BASE_URL)
text_response = client.responses.create(
    model=MODEL_NAME,
    input="Respond with ready",
    reasoning={"effort": "low"},
    max_output_tokens=64,
)
print(f"Text response:\n{text_response.output_text}")

In [ ]:
shell_tool = {
    "type": "function",
    "name": "shell",
    "description": "Runs a command in the user's current shell working directory.",
    "parameters": {
        "type": "object",
        "properties": {"command": {"type": "string"}},
        "required": ["command"],
        "additionalProperties": False,
    },
    "strict": True,
}
tool_response = client.responses.create(
    model=MODEL_NAME,
    input="List the files in the current directory.",
    tools=[shell_tool],
    tool_choice={"type": "function", "name": "shell"},
    parallel_tool_calls=False,
    max_output_tokens=128,
)
calls = [item for item in tool_response.output if item.type == "function_call"]
print(f"Tool call: {calls[0].name}")
print(json.dumps(json.loads(calls[0].arguments), indent=2))

## Connect `ask`

Copy the printed endpoint URL and model name into the shell where you run `ask`.

In [ ]:
print(f'export ASK_BASE_URL="{BASE_URL}"')
print('export ASK_API_KEY="$HF_TOKEN"')
print(f'export ASK_MODEL="{MODEL_NAME}"')
print('export ASK_MAX_OUTPUT_TOKENS="512"')

## Pause the endpoint

Run this after every test session. Pausing stops GPU billing while preserving the endpoint configuration and URL. Rerun the start cell to resume it.

In [ ]:
endpoint.pause()
print(f"Paused {endpoint.name}")